In [6]:
!pip install -q trl transformers datasets peft bitsandbytes accelerate

import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# ==========================================
# 1. КОНФИГУРАЦИЯ И ЗАГРУЗКА ДАННЫХ (ОБНОВЛЕННАЯ ТОКЕНИЗАЦИЯ)
# ==========================================
MODEL_NAME = "Iker/ClickbaitFighter-7B"
DATA_PATH = "Data_clickbait_1_balanced_ready.csv"
OUTPUT_DIR = "./clickbait_fighter_ru_lora"

df = pd.read_csv(DATA_PATH).dropna(subset=['headline', 'text', 'answer_summary'])

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Функция форматирования и принудительной токенизации (задает лимит 1024)
def format_and_tokenize(row):
    messages = [
        {
            "role": "system",
            "content": "Ты — ИИ ClickbaitFighter. Прочитай кликбейтный заголовок и текст, выдай один короткий ответ на русском языке, раскрывающий интригу."
        },
        {
            "role": "user",
            "content": f"Заголовок: {row['headline']}\n\nТекст статьи:\n{row['text']}"
        },
        {
            "role": "assistant",
            "content": str(row['answer_summary'])
        }
    ]
    # Применяем официальный шаблон чата модели и токенизируем с обрезкой
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    tokenized = tokenizer(text, max_length=1024, truncation=True)
    return {"input_ids": tokenized["input_ids"], "attention_mask": tokenized["attention_mask"]}

dataset = Dataset.from_pandas(df)
dataset = dataset.map(format_and_tokenize, remove_columns=dataset.column_names)

# ==========================================
# 2. ЗАГРУЗКА МОДЕЛИ И ТОКЕНИЗАТОРА (4-bit)
# ==========================================
# Настройка квантования для экономии T4 GPU в Colab
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

# 1. Подготовка подгруженных 4-bit весов к обучению
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# !!! ВОТ ЭТОЙ СТРОЧКИ НЕ ХВАТАЛО !!!
# 2. Создаем PEFT-модель и внедряем в нее LoRA адаптеры
model = get_peft_model(model, lora_config)

# Полезно вывести в консоль процент обучаемых параметров, чтобы убедиться, что всё ок
model.print_trainable_parameters()
# ==========================================
# 3. НАСТРОЙКА ОБУЧЕНИЯ (ИСПОЛЬЗУЕМ СТАБИЛЬНЫЙ TRAINER)
# ==========================================
from transformers import Trainer, DataCollatorForLanguageModeling # Меняем импорт

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=2,
    bf16=False,
    fp16=True,
    optim="paged_adamw_8bit",
    save_strategy="epoch",
    report_to="none"
)

# Используем стандартный Trainer, так как данные уже токенизированы
trainer = Trainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False) # Сборщик батчей
)

# Запуск обучения
print("🚀 Начинаем дообучение на русском датасете...")
trainer.train()

# Сохраняем веса LoRA-адаптера и токенизатор
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Обучение завершено! Адаптер сохранен в {OUTPUT_DIR}")

Map:   0%|          | 0/538 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 41,943,040 || all params: 7,283,691,520 || trainable%: 0.5758
🚀 Начинаем дообучение на русском датасете...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.567056
20,1.354729
30,1.395543
40,1.363531
50,1.373353
60,1.320808
70,1.301917
80,1.028661
90,1.015624
100,1.034093


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


✅ Обучение завершено! Адаптер сохранен в ./clickbait_fighter_ru_lora


In [1]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# ==========================================
# 1. ПУТИ И КОНФИГУРАЦИЯ
# ==========================================
BASE_MODEL_NAME = "Iker/ClickbaitFighter-7B"
LORA_ADAPTER_DIR = "./clickbait_fighter_ru_lora"  # Папка, куда сохранился адаптер

# Загружаем токенизатор из папки с адаптером
tokenizer = AutoTokenizer.from_pretrained(LORA_ADAPTER_DIR)

# Настройка 4-бит для экономии памяти на T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("⏳ Загружаем базовую модель...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

print("⚡ Подключаем обученный LoRA-адаптер...")
model = PeftModel.from_pretrained(base_model, LORA_ADAPTER_DIR)
model.eval()  # Переводим модель в режим предсказания

# ==========================================
# 2. ФУНКЦИЯ ДЛЯ ГЕНЕРАЦИИ ОТВЕТА
# ==========================================
def predict_anti_clickbait(headline, text):
    messages = [
        {
            "role": "system",
            "content": "Ты — ИИ ClickbaitFighter. Прочитай кликбейтный заголовок и текст, выдай один короткий ответ на русском языке, раскрывающий интригу."
        },
        {
            "role": "user",
            "content": f"Заголовок: {headline}\n\nТекст статьи:\n{text}"
        }
    ]

    # Применяем шаблон чата, но БЕЗ ассистента (модель сама должна продолжить)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
    **inputs,
    max_new_tokens=65,             # Короткий лимит заставит модель не лить воду
    do_sample=False,               # !!! ОТКЛЮЧАЕМ СЛУЧАЙНОСТЬ (уберет опечатки и суржик)
    repetition_penalty=1.03,       # Минимальный штраф (только чтобы не зацикливалась на знаках)
    no_repeat_ngram_size=3,        # Защита от повторения одинаковых фраз из 3 слов
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id
)

    # Декодируем только то, что сгенерировала модель (отрезаем сам промпт)
    generated_tokens = outputs[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

# ==========================================
# 3. ТЕСТ И ЗАПУСК НА ПРИМЕРЕ
# ==========================================
print("\n--- Проверка на тестовом примере ---")
test_headline = "Вы не поверите, что этот копеечный овощ делает со спиной!"
test_text = "Многие жалуются на боли в пояснице после долгого рабочего дня. Оказывается, обычная свекла содержит элементы, которые снимают воспаление. Достаточно добавить её в рацион два раза в неделю, и боли уменьшатся."

prediction = predict_anti_clickbait(test_headline, test_text)
print(f"Кликбейт: {test_headline}")
print(f"Ответ модели: {prediction}")

# Загружаем датасет, для которого нужны предсказания
df_test = pd.read_csv("Data_clickbait_1_balanced_ready.csv").head(10) # Возьмем первые 10 для теста

print("\nЗапуск массового предсказания...")
df_test['predicted_answer'] = df_test.apply(
    lambda row: predict_anti_clickbait(row['headline'], row['text']), axis=1
)

# Сохраняем результат
df_test.to_csv("predictions_results_7.csv", index=False)
print("✅ Результаты успешно сохранены в файл predictions_results.csv")

⏳ Загружаем базовую модель...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


⚡ Подключаем обученный LoRA-адаптер...

--- Проверка на тестовом примере ---


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Кликбейт: Вы не поверите, что этот копеечный овощ делает со спиной!
Ответ модели: Свекла содражает элемент, который снимает воспаления в позвоночнике. Добавляя ее в рацион два раza в неднелю, боли будут уменшаться. Кроме того, свекла помога

Запуск массового предсказания...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✅ Результаты успешно сохранены в файл predictions_results.csv


In [4]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# ==========================================
# 1. ПУТИ И КОНФИГУРАЦИЯ
# ==========================================
BASE_MODEL_NAME = "Iker/ClickbaitFighter-7B"
LORA_ADAPTER_DIR = "./clickbait_fighter_ru_lora"  # Папка, куда сохранился адаптер

# Загружаем токенизатор из папки с адаптером
tokenizer = AutoTokenizer.from_pretrained(LORA_ADAPTER_DIR)

# Настройка 4-бит для экономии памяти на T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("⏳ Загружаем базовую модель...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

print("⚡ Подключаем обученный LoRA-адаптер...")
model = PeftModel.from_pretrained(base_model, LORA_ADAPTER_DIR)
model.eval()  # Переводим модель в режим предсказания

# ==========================================
# 2. ФУНКЦИЯ ДЛЯ ГЕНЕРАЦИИ ОТВЕТА
# ==========================================
def predict_anti_clickbait(headline, text):
    messages = [
        {
            "role": "system",
            "content": "Ты — ИИ ClickbaitFighter. Прочитай кликбейтный заголовок и текст, выдай один короткий ответ на русском языке, раскрывающий интригу."
        },
        {
            "role": "user",
            "content": f"Заголовок: {headline}\n\nТекст статьи:\n{text}"
        }
    ]

    # Применяем шаблон чата, но БЕЗ ассистента (модель сама должна продолжить)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
    **inputs,
    max_new_tokens=60,             # Уменьшаем, так как ответ должен быть коротким
    do_sample=True,
    temperature=0.4,               # Чуть поднимаем, чтобы текст был более живым и естественным
    top_p=0.85,
    repetition_penalty=1.2,        # !!! ШТРАФ ЗА ПОВТОРЫ: уберет «Ранее сообщалось» и зацикливание
    no_repeat_ngram_size=3,        # Запрещает повторять одинаковые цепочки из 3 слов
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id
)

    # Декодируем только то, что сгенерировала модель (отрезаем сам промпт)
    generated_tokens = outputs[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

# ==========================================
# 3. ТЕСТ И ЗАПУСК НА ПРИМЕРЕ
# ==========================================
print("\n--- Проверка на тестовом примере ---")
test_headline = "Вы не поверите, что этот копеечный овощ делает со спиной!"
test_text = "Многие жалуются на боли в пояснице после долгого рабочего дня. Оказывается, обычная свекла содержит элементы, которые снимают воспаление. Достаточно добавить её в рацион два раза в неделю, и боли уменьшатся."

prediction = predict_anti_clickbait(test_headline, test_text)
print(f"Кликбейт: {test_headline}")
print(f"Ответ модели: {prediction}")

# Загружаем датасет, для которого нужны предсказания
df_test = pd.read_csv("Data_clickbait_1_balanced_ready.csv").head(10) # Возьмем первые 10 для теста

print("\nЗапуск массового предсказания...")
df_test['predicted_answer'] = df_test.apply(
    lambda row: predict_anti_clickbait(row['headline'], row['text']), axis=1
)

# Сохраняем результат
df_test.to_csv("predictions_results_fin.csv", index=False)
print("✅ Результаты успешно сохранены в файл predictions_results_fin.csv")

⏳ Загружаем базовую модель...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


⚡ Подключаем обученный LoRA-адаптер...

--- Проверка на тестовом примере ---


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Кликбейт: Вы не поверите, что этот копеечный овощ делает со спиной!
Ответ модели: Свекла содраживает воспаления в позвоночнике. Главное – есть ее регулярно. Раз в две недели. Помогает также аппетит. Можно готовить разные б

Запуск массового предсказания...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✅ Результаты успешно сохранены в файл predictions_results_2_2.csv


In [1]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from tqdm import tqdm  # Добавляем для отслеживания прогресса 538 новостей

# Очищаем кэш памяти перед большим запуском
torch.cuda.empty_cache()

# ==========================================
# 1. ПУТИ И КОНФИГУРАЦИЯ
# ==========================================
BASE_MODEL_NAME = "Iker/ClickbaitFighter-7B"
LORA_ADAPTER_DIR = "./clickbait_fighter_ru_lora"  # Папка, куда сохранился адаптер

# Загружаем токенизатор из папки с адаптером
tokenizer = AutoTokenizer.from_pretrained(LORA_ADAPTER_DIR)

# Настройка 4-бит для экономии памяти на T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("⏳ Загружаем базовую модель...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

print("⚡ Подключаем обученный LoRA-адаптер...")
model = PeftModel.from_pretrained(base_model, LORA_ADAPTER_DIR)
model.eval()  # Переводим модель в режим предсказания

# ==========================================
# 2. ФУНКЦИЯ ДЛЯ ГЕНЕРАЦИИ ОТВЕТА (БЕЗ ИЗМЕНЕНИЙ)
# ==========================================
def predict_anti_clickbait(headline, text):
    messages = [
        {
            "role": "system",
            "content": "Ты — ИИ ClickbaitFighter. Прочитай кликбейтный заголовок и текст, выдай один короткий ответ на русском языке, раскрывающий интригу."
        },
        {
            "role": "user",
            "content": f"Заголовок: {headline}\n\nТекст статьи:\n{text}"
        }
    ]

    # Применяем шаблон чата, но БЕЗ ассистента (модель сама должна продолжить)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,             # Уменьшаем, так как ответ должен быть коротким
            do_sample=True,
            temperature=0.4,               # Чуть поднимаем, чтобы текст был более живым и естественным
            top_p=0.85,
            repetition_penalty=1.2,        # !!! ШТРАФ ЗА ПОВТОРЫ: уберет «Ранее сообщалось» и зацикливание
            no_repeat_ngram_size=3,        # Запрещает повторять одинаковые цепочки из 3 слов
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    # Декодируем только то, что сгенерировала модель (отрезаем сам промпт)
    generated_tokens = outputs[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

# ==========================================
# 3. ТЕСТ И МАССОВЫЙ ЗАПУСК НА ВСЕМ ДАТАСЕТЕ
# ==========================================
print("\n--- Проверка на тестовом примере ---")
test_headline = "Вы не поверите, что этот копеечный овощ делает со спиной!"
test_text = "Многие жалуются на боли в пояснице после долгого рабочего дня. Оказывается, обычная свекла содержит элементы, которые снимают воспаление. Достаточно добавить её в рацион два раза в неделю, и боли уменьшатся."

prediction = predict_anti_clickbait(test_headline, test_text)
print(f"Кликбейт: {test_headline}")
print(f"Ответ модели: {prediction}")

# ЗАГРУЖАЕМ ВЕСЬ ДАТАСЕТ (убрали .head(10))
print("\nЗагрузка полного датасета...")
df_test = pd.read_csv("Data_clickbait_1_balanced_ready.csv")
print(f"Успешно загружено строк для обработки: {len(df_test)}")

# Инициализируем прогресс-бар для принудительного контроля времени
tqdm.pandas(desc="Обработка новостей")

print("\nЗапуск массового предсказания (это займет около 15-25 минут)...")
# Вместо .apply используем .progress_apply, чтобы видеть бегущую строку процентов
df_test['predicted_answer'] = df_test.progress_apply(
    lambda row: predict_anti_clickbait(row['headline'], row['text']), axis=1
)

# Сохраняем результат
output_filename = "predictions_results_full_dataset.csv"
df_test.to_csv(output_filename, index=False)
print(f"\n✅ Все 538 результатов успешно сохранены в файл: {output_filename}")

# Автоматический триггер скачивания готового файла на ваш ПК (чтобы Colab его не стер)
try:
    from google.colab import files
    files.download(output_filename)
    print("📥 Файл отправлен на скачивание в ваш браузер!")
except Exception as e:
    print("Не удалось запустить автоскачивание, файл доступен в боковой панели Colab.")

⏳ Загружаем базовую модель...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


⚡ Подключаем обученный LoRA-адаптер...

--- Проверка на тестовом примере ---


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Кликбейт: Вы не поверите, что этот копеечный овощ делает со спиной!
Ответ модели: Свекла помогает избавляться от болей в позвоночнике благодаря своим антивоспалительным свойствам. Ее следует есть дважды в неdelee. Подробнее... А также

Загрузка полного датасета...
Успешно загружено строк для обработки: 538

Запуск массового предсказания (это займет около 15-25 минут)...


Обработка новостей:   0%|          | 0/538 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Обработка новостей:  22%|██▏       | 119/538 [25:18<1:29:07, 12.76s/it]


OutOfMemoryError: CUDA out of memory. Tried to allocate 7.67 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.29 GiB is free. Including non-PyTorch memory, this process has 13.27 GiB memory in use. Of the allocated memory 12.70 GiB is allocated by PyTorch, and 452.30 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [1]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from tqdm import tqdm
import os
import gc

# ==========================================
# 1. ПУТИ И КОНФИГУРАЦИЯ
# ==========================================
BASE_MODEL_NAME = "Iker/ClickbaitFighter-7B"
LORA_ADAPTER_DIR = "./clickbait_fighter_ru_lora"
OUTPUT_FILENAME = "predictions_results_fin.csv"

tokenizer = AutoTokenizer.from_pretrained(LORA_ADAPTER_DIR)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("⏳ Загружаем базовую модель...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

print("⚡ Подключаем обученный LoRA-адаптер...")
model = PeftModel.from_pretrained(base_model, LORA_ADAPTER_DIR)
model.eval()

# ==========================================
# 2. ФУНКЦИЯ ДЛЯ ГЕНЕРАЦИИ ОТВЕТА
# ==========================================
def predict_anti_clickbait(headline, text):
    # Защита от OOM: обрезаем слишком длинный исходный текст статьи (максимум ~300-400 слов)
    # Этого более чем достаточно, чтобы понять суть, и спасет GPU от перегрузки
    short_text = str(text)[:1200]

    messages = [
        {
            "role": "system",
            "content": "Ты — ИИ ClickbaitFighter. Прочитай кликбейтный заголовок и текст, выдай один короткий ответ на русском языке, раскрывающий интригу."
        },
        {
            "role": "user",
            "content": f"Заголовок: {headline}\n\nТекст статьи:\n{short_text}"
        }
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=True,
            temperature=0.4,
            top_p=0.85,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs['input_ids'].shape[-1]:]
    result = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # Жесткая очистка памяти после генерации ОДНОЙ строки
    del inputs, outputs, generated_tokens
    torch.cuda.empty_cache()
    gc.collect()

    return result

# ==========================================
# 3. БЕЗОПАСНЫЙ ЦИКЛ ОБРАБОТКИ (Идем по строкам)
# ==========================================
print("\nЗагрузка полного датасета...")
df_test = pd.read_csv("Data_clickbait_1_balanced_ready.csv")
print(f"Всего строк для обработки: {len(df_test)}")

# Проверяем, есть ли уже частично записанный файл (контрольная точка восстановления)
if os.path.exists(OUTPUT_FILENAME):
    print(f"Найдена предыдущая копия файла {OUTPUT_FILENAME}. Загружаем прогресс...")
    df_progress = pd.read_csv(OUTPUT_FILENAME)
    processed_count = len(df_progress)
    print(f"Уже успешно обработано: {processed_count} строк. Продолжаем с нее...")
    # Копируем уже готовые ответы
    df_test.loc[:processed_count-1, 'predicted_answer'] = df_progress['predicted_answer'].values
    start_idx = processed_count
else:
    df_test['predicted_answer'] = None
    start_idx = 0

print(f"\nЗапуск массового предсказания со строки №{start_idx}...")

# Запускаем цикл с контролем очистки памяти и автосохранением
for idx in tqdm(range(start_idx, len(df_test)), desc="Обработка новостей"):
    row = df_test.iloc[idx]

    try:
        ans = predict_anti_clickbait(row['headline'], row['text'])
        df_test.at[idx, 'predicted_answer'] = ans
    except Exception as e:
        print(f"\nОшибка на строке {idx}: {e}")
        print("Сохраняем то, что успели сделать, и останавливаемся.")
        df_test.iloc[:idx].to_csv(OUTPUT_FILENAME, index=False)
        break

    # Каждые 10 строк сохраняем промежуточный результат на диск (на всякий случай)
    if idx % 10 == 0 or idx == len(df_test) - 1:
        df_test.iloc[:idx+1].to_csv(OUTPUT_FILENAME, index=False)

print(f"\n✅ Все доступные результаты сохранены в файл: {OUTPUT_FILENAME}")

# Финальное скачивание в браузер
try:
    from google.colab import files
    files.download(OUTPUT_FILENAME)
    print("📥 Файл автоматически отправлен на скачивание!")
except:
    pass

⏳ Загружаем базовую модель...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


⚡ Подключаем обученный LoRA-адаптер...

Загрузка полного датасета...
Всего строк для обработки: 538

Запуск массового предсказания со строки №0...


Обработка новостей:   0%|          | 0/538 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Обработка новостей: 100%|██████████| 538/538 [1:43:33<00:00, 11.55s/it]


✅ Все доступные результаты сохранены в файл: predictions_results_fin.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Файл автоматически отправлен на скачивание!
